In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

In [2]:
load_dotenv("../.env")
census_key = os.getenv("CENSUS_API_KEY")

In [3]:
years = [2018, 2019, 2020, 2021, 2022, 2023]

all_dfs = []

In [4]:
for year in years:
    url = f"https://api.census.gov/data/{year}/acs/acs5/subject"

    params = {
        "get": "NAME,S1501_C02_008E,S1501_C02_009E,S1501_C02_015E",
        "for": "county:*",
        "in": "state:54",   # West Virginia only
        "key": census_key
    }

    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame(data[1:], columns=data[0])

    df = df.rename(columns={
        "S1501_C02_008E": "Pct_Less_Than_HS",
        "S1501_C02_009E": "Pct_HS_Grad",
        "S1501_C02_015E": "Pct_Bachelors_Plus"
    })

    df["year"] = year

    # Create full FIPS
    df["FIPS"] = df["state"] + df["county"]

    all_dfs.append(df)

education_panel = pd.concat(all_dfs, ignore_index=True)

education_panel

,NAME,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,state,county,year,FIPS
0,"Grant County, West Virginia",10.5,45.3,14.4,54,023,2018,54023
1,"Hampshire County, West Virginia",13.5,44.9,12.2,54,027,2018,54027
2,"Brooke County, West Virginia",6.1,41.7,19.4,54,009,2018,54009
3,"Doddridge County, West Virginia",12.0,45.1,15.7,54,017,2018,54017
4,"Hardy County, West Virginia",12.6,46.4,14.5,54,031,2018,54031
...,...,...,...,...,...,...,...,...
325,"Webster County, West Virginia",9.5,49.2,12.6,54,101,2023,54101
326,"Wetzel County, West Virginia",7.5,48.6,13.5,54,103,2023,54103
327,"Wirt County, West Virginia",9.6,47.9,16.2,54,105,2023,54105
328,"Wood County, West Virginia",7.0,35.5,22.9,54,107,2023,54107


In [5]:
education_panel["FIPS_Code"] = education_panel["state"] + education_panel["county"]
education_panel.drop(columns=["state", "county", "FIPS"], inplace=True)
education_panel

,NAME,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,year,FIPS_Code
0,"Grant County, West Virginia",10.5,45.3,14.4,2018,54023
1,"Hampshire County, West Virginia",13.5,44.9,12.2,2018,54027
2,"Brooke County, West Virginia",6.1,41.7,19.4,2018,54009
3,"Doddridge County, West Virginia",12.0,45.1,15.7,2018,54017
4,"Hardy County, West Virginia",12.6,46.4,14.5,2018,54031
...,...,...,...,...,...,...
325,"Webster County, West Virginia",9.5,49.2,12.6,2023,54101
326,"Wetzel County, West Virginia",7.5,48.6,13.5,2023,54103
327,"Wirt County, West Virginia",9.6,47.9,16.2,2023,54105
328,"Wood County, West Virginia",7.0,35.5,22.9,2023,54107


In [6]:
education_panel["County"] = education_panel["NAME"].apply(lambda x: x.split(",")[0])
education_panel.drop(columns=["NAME"], inplace=True)
education_panel

,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,year,FIPS_Code,County
0,10.5,45.3,14.4,2018,54023,Grant County
1,13.5,44.9,12.2,2018,54027,Hampshire County
2,6.1,41.7,19.4,2018,54009,Brooke County
3,12.0,45.1,15.7,2018,54017,Doddridge County
4,12.6,46.4,14.5,2018,54031,Hardy County
...,...,...,...,...,...,...
325,9.5,49.2,12.6,2023,54101,Webster County
326,7.5,48.6,13.5,2023,54103,Wetzel County
327,9.6,47.9,16.2,2023,54105,Wirt County
328,7.0,35.5,22.9,2023,54107,Wood County


In [7]:
education_panel.rename(columns={"year": "Year"}, inplace=True)
education_panel

,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,Year,FIPS_Code,County
0,10.5,45.3,14.4,2018,54023,Grant County
1,13.5,44.9,12.2,2018,54027,Hampshire County
2,6.1,41.7,19.4,2018,54009,Brooke County
3,12.0,45.1,15.7,2018,54017,Doddridge County
4,12.6,46.4,14.5,2018,54031,Hardy County
...,...,...,...,...,...,...
325,9.5,49.2,12.6,2023,54101,Webster County
326,7.5,48.6,13.5,2023,54103,Wetzel County
327,9.6,47.9,16.2,2023,54105,Wirt County
328,7.0,35.5,22.9,2023,54107,Wood County


In [8]:
education_panel = education_panel[["Year", "FIPS_Code", "County", "Pct_Less_Than_HS", "Pct_HS_Grad", "Pct_Bachelors_Plus"]]
education_panel

,Year,FIPS_Code,County,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus
0,2018,54023,Grant County,10.5,45.3,14.4
1,2018,54027,Hampshire County,13.5,44.9,12.2
2,2018,54009,Brooke County,6.1,41.7,19.4
3,2018,54017,Doddridge County,12.0,45.1,15.7
4,2018,54031,Hardy County,12.6,46.4,14.5
...,...,...,...,...,...,...
325,2023,54101,Webster County,9.5,49.2,12.6
326,2023,54103,Wetzel County,7.5,48.6,13.5
327,2023,54105,Wirt County,9.6,47.9,16.2
328,2023,54107,Wood County,7.0,35.5,22.9


In [9]:
education_panel.to_csv("education_data.csv", index=False)